# 02 - Dataset Review and Split

This notebook continues the project using the **CRISP-DM** workflow.

At this stage, we are still in **Data Understanding** and **Data Preparation**. The goal is to review the labelled dataset created in `01_data_prep.ipynb`, confirm that the labels and image paths look usable, and prepare a clean table that can later be split for model training.

We will keep the notebook simple and readable. Each section reloads what it needs from saved files, so the notebook can be closed, reopened, and continued without rerunning everything before it.

## How we will work in this notebook

We will not write the whole machine learning pipeline at once.

Instead, we will move in small, checkable sections:

1. Load the saved data-prep outputs.
2. Confirm the expected columns exist.
3. Check how many labelled images are usable.
4. Build a clean model-candidate dataset.
5. Save that table for the next notebook section.

After running this notebook section, we will inspect the results before creating the final train, validation, and test split.

## Install required libraries

Run this once if the notebook environment does not already have the required packages.

The project mainly needs `pandas` for tables, `numpy` for small numeric checks, `matplotlib` for charts, `scikit-learn` for splitting later, and `Pillow` for image checks.

In [ ]:
%pip install pandas numpy matplotlib scikit-learn pillow

## Section 1: Set up paths and imports

This cell keeps all project paths in one place.

The notebook expects the processed CSV files from `01_data_prep.ipynb` to already exist in `data/processed/`.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

current_folder = Path.cwd()
PROJECT_ROOT = current_folder.parent if current_folder.name == "notebooks" else current_folder

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR

## Section 2: Load the prepared tables

The first notebook saved two important tables:

- `irr_image_labels.csv`: one row per image/IRR record, with the main label attached.
- `irr_pattern_entries.csv`: one row per pattern entry found in field `9.307`.

For model training, the image-level table is the main table. The pattern-entry table remains useful for checking multi-label or repeated-label cases.

In [ ]:
image_labels_path = PROCESSED_DIR / "irr_image_labels.csv"
pattern_entries_path = PROCESSED_DIR / "irr_pattern_entries.csv"
label_counts_path = PROCESSED_DIR / "pattern_label_counts.csv"

for path in [image_labels_path, pattern_entries_path, label_counts_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing expected file: {path}")

In [ ]:
id_columns = {
    "subject_id": "string",
    "finger_position": "string",
    "resolution": "string",
}

image_labels = pd.read_csv(image_labels_path, dtype=id_columns)
pattern_entries = pd.read_csv(pattern_entries_path, dtype=id_columns)
label_counts = pd.read_csv(label_counts_path)

print("Image-level rows:", len(image_labels))
print("Pattern-entry rows:", len(pattern_entries))
print("Label-count rows:", len(label_counts))

## Section 3: Check the columns we need

Before doing any analysis, we confirm that the table has the fields needed for classification.

The important fields are:

- `primary_label`: the raw SD302 pattern label chosen for the image.
- `broad_class`: the broader class, such as arch, loop, or whorl.
- `subtype`: the subtype where the raw label gives one.
- `png_path`: the linked fingerprint image.
- `png_found`: whether the image file was found in the workspace.

In [ ]:
required_columns = [
    "subject_id",
    "finger_position",
    "primary_label",
    "broad_class",
    "subtype",
    "png_path",
    "png_found",
]

missing_columns = [column for column in required_columns if column not in image_labels.columns]
missing_columns

In [ ]:
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are present.")

## Section 4: Review the available labels

This shows the labels extracted from the raw SD302g records.

The counts help us decide what kind of classifier is realistic. Broad classes should be more stable. Some subtypes have very small counts, so they must be handled carefully.

In [ ]:
label_counts

In [ ]:
broad_class_counts = image_labels["broad_class"].value_counts().sort_index()
broad_class_counts

In [ ]:
ax = broad_class_counts.plot(kind="bar", figsize=(7, 4))
ax.set_title("Available Images by Broad Pattern Class")
ax.set_xlabel("Broad class")
ax.set_ylabel("Number of images")
plt.xticks(rotation=0)
plt.show()

## Section 5: Keep only rows with linked images

A row is useful for image classification only if the image file exists.

Here we create a clean candidate table for modelling. This is not the final training split yet; it is the cleaned table we will inspect first.

In [ ]:
model_candidates = image_labels.copy()

model_candidates = model_candidates[model_candidates["png_found"] == True]
model_candidates = model_candidates.dropna(subset=["png_path", "broad_class"])

print("Usable image rows:", len(model_candidates))

In [ ]:
model_candidates[[
    "subject_id",
    "finger_position",
    "primary_label",
    "broad_class",
    "subtype",
    "png_path",
]].head()

## Section 6: Save the clean model-candidate table

This table becomes the handoff point for the next step.

Saving it means the next notebook section can start from this file directly instead of depending on variables in memory.

In [ ]:
model_candidates_path = PROCESSED_DIR / "model_candidates.csv"

model_candidates.to_csv(model_candidates_path, index=False)

print("Saved:", model_candidates_path)
print("Rows:", len(model_candidates))

## Stop and review

Pause here after running the cells above.

The important things to check are:

- Are all required columns present?
- How many usable image rows were found?
- Are the broad classes balanced enough for a first classifier?
- Does the sample table show realistic image paths and labels?

Once these results look correct, the next section will create a subject-aware train, validation, and test split.

## Section 7: Create the first modelling dataset

For the first classifier, we will use the classes with enough examples to train and evaluate properly:

- `arch`
- `left_slant_loop`
- `right_slant_loop`
- `whorl`

The `unclassifiable` group has only a small number of images, so we will keep it out of the first supervised model. We can return to it later as a reject/unknown class or as a separate analysis point.

In [ ]:
from pathlib import Path

import pandas as pd

current_folder = Path.cwd()
PROJECT_ROOT = current_folder.parent if current_folder.name == "notebooks" else current_folder
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

model_candidates_path = PROCESSED_DIR / "model_candidates.csv"

In [ ]:
id_columns = {
    "subject_id": "string",
    "finger_position": "string",
    "resolution": "string",
}

model_candidates = pd.read_csv(model_candidates_path, dtype=id_columns)

print("Loaded rows:", len(model_candidates))

In [ ]:
broad_classes_for_first_model = [
    "arch",
    "left_slant_loop",
    "right_slant_loop",
    "whorl",
]

broad_model_data = model_candidates[
    model_candidates["broad_class"].isin(broad_classes_for_first_model)
].copy()

broad_model_data["broad_class"].value_counts().sort_index()

## Section 8: Split by subject, not by image

This is important for academic quality.

If images from the same subject appear in both training and testing, the model may partly learn the person instead of learning the fingerprint pattern. To reduce that risk, we split using `subject_id` as a group.

The split will be:

- 70% training
- 15% validation
- 15% testing

In [ ]:
%pip install scikit-learn

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

first_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42,
)

train_index, temp_index = next(
    first_split.split(broad_model_data, groups=broad_model_data["subject_id"])
)

In [ ]:
train_data = broad_model_data.iloc[train_index].copy()
temp_data = broad_model_data.iloc[temp_index].copy()

second_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42,
)

validation_index, test_index = next(
    second_split.split(temp_data, groups=temp_data["subject_id"])
)

In [ ]:
validation_data = temp_data.iloc[validation_index].copy()
test_data = temp_data.iloc[test_index].copy()

train_data["split"] = "train"
validation_data["split"] = "validation"
test_data["split"] = "test"

split_data = pd.concat([train_data, validation_data, test_data], ignore_index=True)

split_data["split"].value_counts()

## Section 9: Check the split quality

After splitting, we check two things:

1. No subject appears in more than one split.
2. Each split still contains examples from all target classes.

In [ ]:
subject_split_counts = split_data.groupby("subject_id")["split"].nunique()
leaked_subjects = subject_split_counts[subject_split_counts > 1]

print("Subjects appearing in more than one split:", len(leaked_subjects))

In [ ]:
class_counts_by_split = pd.crosstab(
    split_data["broad_class"],
    split_data["split"],
)

class_counts_by_split

## Section 10: Save the split dataset

This saved file is the starting point for model training.

The next notebook can load this file directly and train the first broad fingerprint pattern classifier.

In [ ]:
split_data_path = PROCESSED_DIR / "broad_model_split.csv"

split_data.to_csv(split_data_path, index=False)

print("Saved:", split_data_path)
print("Rows:", len(split_data))

## Stop and review before modelling

Pause here after running the split section.

Before training a model, we should confirm:

- `Subjects appearing in more than one split` is `0`.
- all four target classes appear in train, validation, and test.
- the class counts are acceptable for a first academic baseline model.